In [9]:
import pathlib
from argparse import ArgumentParser
import yaml
import torch
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint

In [3]:
import sys 
sys.path.append('../')
from src.attn_tracking_lightning import AttentionalTrackingModule


In [4]:
path = '../config/attentional_cue/attn_cue_lr_1e-4_bs_64.yaml'
config = yaml.load(open(path, 'r'), Loader=yaml.FullLoader)

In [5]:
model = AttentionalTrackingModule(config)

In [76]:
!ls ../attn_cue_models/attn_cue_jsin_pilot_no_pretrain_bs_64_lr_1e-4/checkpoints/

epoch=0-step=14999-v1.ckpt  epoch=0-step=24999.ckpt
epoch=0-step=14999-v2.ckpt  epoch=0-step=34999-v1.ckpt
epoch=0-step=14999.ckpt     epoch=0-step=34999.ckpt
epoch=0-step=19999-v1.ckpt  epoch=0-step=4999-v1.ckpt
epoch=0-step=19999-v2.ckpt  epoch=0-step=9999.ckpt


In [77]:
ckpt_path = '../attn_cue_models/attn_cue_jsin_pilot_no_pretrain_bs_64_lr_1e-4/checkpoints/epoch=0-step=19999-v2.ckpt'

ckpt = torch.load(ckpt_path, map_location=torch.device('cpu'))

model.load_state_dict(ckpt['state_dict'])


<All keys matched successfully>

In [78]:
attn_params = model.attn_modules

In [79]:
[param for param in attn_params[0].named_parameters()]

[('bias',
  Parameter containing:
  tensor([0.8778], requires_grad=True)),
 ('slope',
  Parameter containing:
  tensor([0.0615], requires_grad=True)),
 ('threshold',
  Parameter containing:
  tensor([-0.5825], requires_grad=True))]

In [80]:
attn_params[0].bias.item()

0.8778233528137207

In [81]:
n_attn_blocks = len(attn_params)

In [33]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

## Plot attentional filters

In [37]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [83]:
n_attn_blocks

8

In [ ]:
fig, axs = plt.subplots(n_attn_blocks//2, 2, sharex=True, figsize=(9,6))
axs = axs.ravel()
layer_names = [name for name in model._modules['model']._modules.keys() if 'attn' in name]
# layer_names = ['cochleagram'] + layer_names

x = np.arange(-10,11)

for i in range(n_attn_blocks):
    bias = attn_params[i].bias.item()
    slope = attn_params[i].slope.item()
    threshold = attn_params[i].threshold.item()
    
    axs[i].plot(x, bias + (1-bias) * sigmoid((x + threshold) * slope))
    axs[i].text(10, 0, f"{bias:.2f=}\n{slope:.2f=}\n{threshold:.2f=}")
    axs[i].set_title(layer_names[i])

    
    
# plt.tight_layout()